# 02 — Preparação: divisão sem vazamento, verificação e visualização

Pré-requisito: notebook 01 executado (ou `interim.zip` no Drive).

In [ ]:
# === Configuração: rode sempre que abrir o notebook ===
REPO_URL = "https://github.com/Lucas-Gomes-Alcantara/Rota-Livre-Visao.git"  # <- troque pelo repositório da equipe
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/rota_livre"          # pasta persistente (o disco do Colab é apagado ao desconectar)
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists("/content/Rota-Livre-Visao"):
    !git clone -q $REPO_URL /content/Rota-Livre-Visao
%cd /content/Rota-Livre-Visao
!pip install -q -r requirements.txt

In [ ]:
if not os.path.exists('data/interim/manifest.csv'):
    !unzip -qo $DRIVE/interim.zip -d .
print(open('data/interim/manifest.csv').readline())

## (Depois de anotar) baixar o projeto Mapillary anotado da equipe
Ligue `mapillary_br` em `configs/sources.yaml` e rode a conversão só dessa fonte.

In [ ]:
# Chaves guardadas em "Secrets" do Colab (ícone de chave na barra lateral) — nunca escreva a chave no notebook.
for k in ("ROBOFLOW_API_KEY", "MAPILLARY_TOKEN"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print(f"{k} não configurada (necessária só para a etapa correspondente)")

# !python src/download_data.py roboflow-own --workspace SEU_WORKSPACE --project SEU_PROJETO --version 1
# !python src/convert.py --only mapillary_br

## 1. Divisão treino/validação/teste por grupo (70/15/15)
Grupos = foto original + cópias aumentadas, sequência Mapillary, quadros consecutivos. Quase-duplicatas (dHash) unem grupos.

In [ ]:
!python src/split.py --ratios 0.7 0.15 0.15 --hash-threshold 6
!cat configs/data.yaml

## 2. Verificação de qualidade

In [ ]:
!python src/check_dataset.py --strict

## 3. Inspeção visual das anotações
**Confira principalmente `desnivel`** (vem do rótulo *broken*, mapeado provisoriamente).

In [ ]:
!python src/visualize_labels.py --per-class 12
from IPython.display import Image as I, display
import glob
display(I('results/figures/distribuicao_classes.png'))
for f in sorted(glob.glob('results/figures/anotacoes_train_*.jpg')):
    print(f); display(I(f, width=900))

In [ ]:
!zip -qr $DRIVE/processed.zip data/processed configs/data.yaml configs/data_test_br.yaml results
print('ok')